
# Analyse de Données Audio
**J3 PM | Traitement de données audio, transcription et automatisation**

**Objectifs d'apprentissage :**
- Transcrire des fichiers audio directement depuis le web sans les télécharger sur le disque.
- Utiliser un LLM pour nettoyer et formater un entretien qualitatif brut.
- Construire une boucle d'automatisation pour traiter un corpus de plusieurs audios (Topic Modeling).

**Prérequis :**
- Une clé API OpenAI configurée.


In [ ]:

!pip install requests openai pandas


In [ ]:

import os
import requests
from io import BytesIO
from openai import OpenAI

# 1. Coller votre clé API d'OPEN AI
api_key = "sk-votre-cle-secrete-ici"

try:
    client = OpenAI(api_key=api_key)
    print("Client OpenAI initialisé avec succès !")
except Exception as e:
    print("Erreur d'initialisation :", e)



## 1. Retranscription : Le State of the Union (Trump)

Nous allons charger un court extrait d'un State of the Union (SOTU) de Donald Trump directement en mémoire, puis utiliser Whisper pour la retranscription en anglais.
*(Source : Archive publique YouTube, State of the Union Address 2018)*


In [ ]:

# Extrait d'un discours de Donald Trump en anglais (State of the Union court)
trump_file = "trump_sotu.mp3"
audio_trump_url = f"https://raw.githubusercontent.com/mickaeltemporao/lillelms/main/resources/j3/2_aprem/audio/{trump_file}"

response = requests.get(audio_trump_url)


In [ ]:

# Certaines API s'attendent à lire un fichier avec un nom explicite (pour déduire le format)
audio = BytesIO(response.content)
audio.name = trump_file

transcript_trump = client.audio.transcriptions.create(
    model="whisper-1",
    file=audio
)

print("--- Retranscription (Anglais) ---")
print(transcript_trump.text)



## 2. Retranscription : Campagne Électorale Française

Analysons maintenant un discours de campagne en français. Pour cet exercice, nous allons retranscrire un extrait d'un meeting de campagne présidentielle de Jean-Luc Mélenchon en utilisant la même méthode.
*(Source : Chaîne YouTube officielle, Meeting de campagne présidentielle 2022)*


In [ ]:

melenchon_file = "melenchon_campagne.mp3"
audio_fr_url = f"https://raw.githubusercontent.com/mickaeltemporao/lillelms/main/resources/j3/2_aprem/audio/{melenchon_file}"

response_fr = requests.get(audio_fr_url)

audio_fr = BytesIO(response_fr.content)
audio_fr.name = melenchon_file

transcript_fr = client.audio.transcriptions.create(
    model="whisper-1",
    file=audio_fr
)

print("--- Retranscription (Français) ---")
print(transcript_fr.text)



## 3. Analyse d'Entretiens Qualitatifs en Sciences Sociales

L'analyse d'entretiens semi-directifs est au cœur de nombreuses recherches qualitatives. Pour illustrer notre méthode, nous utiliserons un court extrait d'une **interview publique du sociologue Pierre Bourdieu** simulant un enregistrement de terrain.
*(Source : Extrait d'interview télévisée, archives YouTube)*


In [ ]:

interview_file = "interview.mp3"
audio_interview_url = f"https://raw.githubusercontent.com/mickaeltemporao/lillelms/main/resources/j3/2_aprem/audio/{interview_file}"

response_interview = requests.get(audio_interview_url)

audio_interview = BytesIO(response_interview.content)
audio_interview.name = interview_file

transcript_interview = client.audio.transcriptions.create(
    model="whisper-1",
    file=audio_interview
)

print("--- Retranscription brute (Interview) ---")
print(transcript_interview.text)



### 3.1 Restructuration par un LLM (Prompt Engineering)

Une transcription Whisper brute comporte souvent de nombreuses hésitations orales ("euh", bégaiements). Nous allons utiliser le LLM pour transformer cet output en un format exploitable (Enquêteur / Enquêté) et corriger les scories de l'oralité.


In [ ]:

prompt_system_clean = '''
Tu es un assistant de recherche en sociologie.
Voici une transcription brute d'un entretien qualitatif.
Ton rôle est de :
1. Distinguer les prises de parole (utilise 'Enquêteur :' et 'Enquêté :').
2. Nettoyer les hésitations (les "euh", les mots répétés) pour rendre la lecture fluide.
3. Ajouter des sauts de ligne entre chaque prise de parole.
'''

reponse_clean = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": prompt_system_clean},
        {"role": "user", "content": f"Voici la retranscription brute :

{transcript_interview.text}"}
    ],
    temperature=0.2
)

interview_propre = reponse_clean.choices[0].message.content
print("--- Entretien nettoyé et structuré ---")
print(interview_propre)



## 4. Automatisation sur un corpus (Topic Modeling)

En recherche, nous avons souvent des dizaines d'audios à traiter. Plutôt que de répéter le code manuellement, l'idéal est de créer une **fonction réutilisable** qui effectue tout le travail (téléchargement, retranscription et extraction des thèmes). Nous pourrons ensuite appeler cette fonction sur tous les fichiers de notre corpus grâce à une simple boucle !


In [ ]:

def analyser_audio_themes(nom_fichier):
    """
    Cette fonction prend en entrée le nom d'un fichier audio, 
    le télécharge en mémoire, le retranscrit avec Whisper, 
    et extrait ses grands thèmes avec GPT-4o-mini.
    """
    base_url = "https://raw.githubusercontent.com/mickaeltemporao/lillelms/main/resources/j3/2_aprem/audio/"
    
    # 1. Téléchargement direct en mémoire
    reponse = requests.get(f"{base_url}{nom_fichier}")
    audio_memoire = BytesIO(reponse.content)
    audio_memoire.name = nom_fichier
    
    # 2. Retranscription avec Whisper
    transcript = client.audio.transcriptions.create(
        model="whisper-1",
        file=audio_memoire
    )
    texte_brut = transcript.text
    print(f"[+] Retranscription de {nom_fichier} terminée ({len(texte_brut)} caractères).")
    
    # 3. Extraction thématique avec le LLM
    prompt_system_themes = '''
    Tu es un chercheur en sciences sociales.
    Identifie les 2 thèmes principaux abordés dans ce court extrait.
    Pour chaque thème, donne simplement un titre court et une phrase d'explication.
    '''
    
    reponse_themes = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": prompt_system_themes},
            {"role": "user", "content": texte_brut}
        ],
        temperature=0.3
    )
    
    return reponse_themes.choices[0].message.content



Maintenant que notre fonction est prête, il suffit d'itérer sur notre liste de fichiers pour tout automatiser.


In [ ]:

# Liste de nos fichiers audio (notre "corpus")
fichiers_audio = ["trump_sotu.mp3", "melenchon_campagne.mp3", "interview.mp3"]

for fichier in fichiers_audio:
    print(f"\n========== TRAITEMENT DE : {fichier} ==========")
    themes = analyser_audio_themes(fichier)
    
    print("[+] Thèmes identifiés :")
    print(themes)
    print("====================================================")
